# WooCommerce Supplier Repor products -- Connecting to Your Database

This notebook connects directly to your WordPress/WooCommerce MySQL database, pulls
sales data into a pandas DataFrame, and lets you analyze it in Python -- automating
the same reconciliation the data for create report.

**What you'll learn in this notebook:**
1. Connecting Python to a remote MySQL database
2. Handling credentials safely (never hardcode passwords!)
3. Running SQL queries and loading results into pandas
4. Exploring a DataFrame
5. A basic chart
6. Exporting results

## Step 1 -- Install the libraries we need
- **pymysql** -- lets Python talk to MySQL (pure Python, nothing extra to compile)
- **sqlalchemy** -- gives pandas a standard, clean way to connect to a database
- **pandas** -- turns SQL results into a DataFrame you can filter, group, and plot
- **python-dotenv** -- loads secret credentials from a separate file, so they never
  end up written inside this notebook
- **matplotlib** -- for the chart in Step 8

In [13]:
#This is the command
!pip install pymysql sqlalchemy pandas python-dotenv matplotlib


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from dotenv import load_dotenv

pd.set_option("display.float_format", lambda x: f"{x:,.0f}")

## Step 2 -- Set up your credentials safely
**Never write your database password directly inside a notebook.** If you ever share
this file, push it to GitHub, or someone looks over your shoulder, that password is
exposed in plain text.

Instead, create a plain text file named `.env` in the **same folder** as this
notebook, with content like this (use your real values):

```
DB_HOST=auth-db1839.hstgr.io
DB_PORT=3306
DB_NAME=your_database_name
DB_USER=your_database_user
DB_PASSWORD=your_database_password
```

If this ever becomes a git project, add a `.gitignore` file containing just `.env`
so it never gets committed by accident.

## 2. Connect to the database

Credentials come from a local `.env` file, never hardcoded here -- see
`.env.example` next to this notebook. Copy it to `.env`, fill in your real
values, and keep `.env` out of version control (add it to `.gitignore`).

If you're running this notebook from your own machine rather than on the
server itself, you'll likely need to enable **Remote MySQL** access for this
database in Hostinger's hPanel and allow your current IP -- shared hosting
only accepts local connections by default.

In [15]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "3306")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

with engine.connect() as conn:
    print("Connected OK")

Connected OK


In [ ]:
## 2. Set the period to check

START_DATE = "2026-08-01"   # inclusive
END_DATE   = "2026-09-01"   # exclusive

#Change these two dates and every query below updates automatically.

query_orders = f"""
SELECT
    p.ID AS order_id,
    p.post_status AS status,
    p.post_date_gmt AS date_created_gmt,
    MAX(CASE WHEN pm.meta_key = '_order_total'    THEN pm.meta_value END) + 0 AS order_total,
    MAX(CASE WHEN pm.meta_key = '_order_tax'      THEN pm.meta_value END) + 0 AS order_tax,
    MAX(CASE WHEN pm.meta_key = '_order_shipping' THEN pm.meta_value END) + 0 AS order_shipping,
    (
        MAX(CASE WHEN pm.meta_key = '_order_total'    THEN pm.meta_value END) -
        MAX(CASE WHEN pm.meta_key = '_order_tax'      THEN pm.meta_value END) -
        MAX(CASE WHEN pm.meta_key = '_order_shipping' THEN pm.meta_value END)
    ) + 0 AS net_sales_recalculated
FROM wp_posts p
JOIN wp_postmeta pm ON pm.post_id = p.ID
WHERE p.post_type = 'shop_order'
  AND p.post_status NOT IN ('wc-pending', 'wc-cancelled', 'wc-failed', 'trash')
  AND p.post_date_gmt BETWEEN '{START_DATE}' AND '{END_DATE}'
GROUP BY p.ID
"""

df_orders = pd.read_sql(query_orders, engine)
df_orders

NameError: name 'START_DATE' is not defined